# Correlation Analysis
## Price Relationships and Dependencies

### Objectives:
- Analyze correlations between OHLCV variables
- Identify leading indicators
- Understand price-volume relationships
- Lag analysis and autocorrelation
- Feature correlation for model development

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Import utilities
import sys
sys.path.append('..')
from utils.data_loader import DataLoader
from utils.visualizations import TradingVisualizer

In [ ]:
# Load data
loader = DataLoader()
df = loader.load_from_csv()

if df.empty:
    df = loader.load_from_db()

print(f"✅ Data loaded: {len(df)} rows")

# Calculate additional metrics
df['returns'] = df['close'].pct_change() * 100
df['range'] = df['high'] - df['low']
df['range_pct'] = (df['high'] - df['low']) / df['close'] * 100
df['mid'] = (df['high'] + df['low']) / 2
df['vwap'] = (df['high'] + df['low'] + df['close']) / 3

df.head()

In [ ]:
# Correlation matrix
corr_cols = ['open', 'high', 'low', 'close', 'volume', 'returns', 'range', 'range_pct', 'mid', 'vwap']
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))

# Heatmap with annotations
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, 
            mask=mask,
            annot=True, 
            fmt='.2f',
            cmap='coolwarm',
            center=0,
            square=True,
            linewidths=0.5,
            cbar_kws={"shrink": 0.8},
            ax=ax)
ax.set_title('Correlation Matrix - OHLCV Variables', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Print correlation summary
print("\n" + "=" * 60)
print("📊 CORRELATION SUMMARY")
print("=" * 60)
print("\nStrong positive correlations (>0.9):")
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        val = corr_matrix.iloc[i, j]
        if val > 0.9:
            print(f"  {corr_matrix.columns[i]:12s} vs {corr_matrix.columns[j]:12s}: {val:.3f}")

In [ ]:
# Volume vs Price relationships
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Volume vs Price
axes[0].scatter(df['volume'], df['close'], alpha=0.5, s=1)
axes[0].set_xlabel('Volume', fontweight='bold')
axes[0].set_ylabel('Price ($)', fontweight='bold')
axes[0].set_title('Volume vs Price', fontsize=12, fontweight='bold')

# Volume vs Returns
axes[1].scatter(df['volume'], df['returns'], alpha=0.5, s=1, color='orange')
axes[1].set_xlabel('Volume', fontweight='bold')
axes[1].set_ylabel('Returns (%)', fontweight='bold')
axes[1].set_title('Volume vs Returns', fontsize=12, fontweight='bold')

# Range vs Volume
axes[2].scatter(df['volume'], df['range_pct'], alpha=0.5, s=1, color='green')
axes[2].set_xlabel('Volume', fontweight='bold')
axes[2].set_ylabel('Range (%)', fontweight='bold')
axes[2].set_title('Volume vs Range (%)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Autocorrelation analysis
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# ACF for returns
plot_acf(df['returns'].dropna(), lags=40, ax=axes[0, 0])
axes[0, 0].set_title('Autocorrelation - Returns', fontsize=12, fontweight='bold')

# PACF for returns
plot_pacf(df['returns'].dropna(), lags=40, ax=axes[0, 1])
axes[0, 1].set_title('Partial Autocorrelation - Returns', fontsize=12, fontweight='bold')

# ACF for log returns
plot_acf(df['log_returns'].dropna(), lags=40, ax=axes[1, 0])
axes[1, 0].set_title('Autocorrelation - Log Returns', fontsize=12, fontweight='bold')

# PACF for log returns
plot_pacf(df['log_returns'].dropna(), lags=40, ax=axes[1, 1])
axes[1, 1].set_title('Partial Autocorrelation - Log Returns', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()